# Parts D and E: figures of the reference run and class-wise metrics of the 10 runs (GROUPED partition)

Uses the `best.pt` of the grouped run of seed 101 (folder `runs_agrupado/run_seed_101/treino/weights/`) and the file list `split_agrupado_seed_101.csv`. **Nothing is trained here**; it runs on GPU or CPU in a few minutes. It requires the outputs of `Rodadas_agrupadas_PLOS_ONE.ipynb`.

It writes, in `runs_agrupado/figuras_seed_101/`:
1. the confusion matrix on the test subset (225 images), as CSV and PNG, and a check that the accuracy is 223/225;
2. the learning curve (training loss, validation loss, and validation accuracy per epoch);
3. Grad-CAM maps (3 correct predictions per cultivar) and, if any, the misclassified images;
4. the share of the Grad-CAM activation inside the brightness mask for the whole test subset (reported for completeness only; the mask is not a stalk segmentation and is not used in the paper).

**Part E (end of the notebook):** for each of the 10 runs, evaluates its `best.pt` on its own test subset and saves the predictions, precision, recall, and F1-score per cultivar, and the inference time (one image per call, `batch=1`). These numbers feed Tables 5 and 6 and the latency paragraph. It also checks that the number of correct predictions of each run agrees with `resumo_por_rodada.csv`.

**Language note.** Comments, messages, and text are in English. File names and column names of the output tables (for example `classe`, `arquivo`, `predicoes_10_rodadas.csv`) are kept as originally generated, so that they match the files deposited with the paper.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q ultralytics==8.4.208 || pip install -q ultralytics

In [ ]:
import os, json, random
import numpy as np, pandas as pd, cv2, torch
import matplotlib; import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO
from ultralytics.data.augment import classify_transforms

# ======================= CONFIGURATION =======================
DATASET_ORIGINAL = '/content/drive/MyDrive/Colab_Notebooks/four_varieties/data_cropped'
BASE_WORK_DIR    = '/content/drive/MyDrive/Colab_Notebooks/four_varieties/runs_agrupado'
SEED = 101                     # reference run of the paper
N_PER_CLASS = 3                # correct predictions per cultivar in the Grad-CAM figure
TARGET_LAYER = 8               # last C2f block of YOLOv8n-cls (before the Classify head)
CROP = 224
# ============================================================
OUT = os.path.join(BASE_WORK_DIR, f'figuras_seed_{SEED}'); os.makedirs(OUT, exist_ok=True)
RUN = os.path.join(BASE_WORK_DIR, f'run_seed_{SEED}')
W = os.path.join(RUN, 'treino', 'weights', 'best.pt')
assert os.path.exists(W), f'best.pt not found: {W}'
S = pd.read_csv(os.path.join(BASE_WORK_DIR, f'split_agrupado_seed_{SEED}.csv'))
TESTE = S[S.particao == 'test'].sort_values(['classe', 'arquivo']).reset_index(drop=True)
CLASSES = sorted(d for d in os.listdir(DATASET_ORIGINAL) if os.path.isdir(os.path.join(DATASET_ORIGINAL, d)) and d != 'inspecao')
print('Classes:', CLASSES, '| test images:', len(TESTE), '| per class:', TESTE.classe.value_counts().sort_index().to_dict())
assert len(TESTE) == 225 and set(TESTE.classe) == set(CLASSES)

yolo = YOLO(W)
net = yolo.model.float().eval()
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; net.to(dev)
NOMES = [yolo.names[i] for i in range(len(yolo.names))]
print('Class order of the model:', NOMES)
assert NOMES == CLASSES, 'The class order of the model differs from CLASSES; fix it before continuing.'
tf = classify_transforms(size=CROP)           # same preprocessing as validation and test

def load_rgb_mask(path):
    im = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if im.ndim == 3 and im.shape[2] == 4:
        mask = im[:, :, 3] > 0; bgr = im[:, :, :3]
    else:
        bgr = im if im.ndim == 3 else cv2.cvtColor(im, cv2.COLOR_GRAY2BGR)
        mask = bgr.max(2) > 5                       # brightness threshold: excludes only near-black pixels (not a stalk segmentation)
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), mask

def center_view(rgb, mask, size=CROP):
    h, w = rgb.shape[:2]; s = size / min(h, w)
    nh, nw = max(size, round(h * s)), max(size, round(w * s))
    im = np.array(Image.fromarray(rgb).resize((nw, nh), Image.BILINEAR))
    mk = np.array(Image.fromarray(mask.astype(np.uint8) * 255).resize((nw, nh), Image.NEAREST)) > 0
    t, l = (nh - size) // 2, (nw - size) // 2
    return im[t:t + size, l:l + size], mk[t:t + size, l:l + size]

def gradcam(path):
    rgb, mask = load_rgb_mask(path)
    cim, cmk = center_view(rgb, mask)
    x = tf(Image.fromarray(rgb)).unsqueeze(0).to(dev).requires_grad_(True)
    store = {}
    h = net.model[TARGET_LAYER].register_forward_hook(lambda m, i, o: store.__setitem__('a', o))
    with torch.enable_grad():
        out = net(x); logits = out[1] if isinstance(out, (tuple, list)) else out
        c = int(logits.argmax(1))
        g = torch.autograd.grad(logits[0, c], store['a'])[0]
    h.remove()
    w = g.mean((2, 3), keepdim=True)
    cam = torch.relu((w * store['a']).sum(1, keepdim=True))
    cam = torch.nn.functional.interpolate(cam, size=(CROP, CROP), mode='bilinear', align_corners=False)[0, 0]
    cam = cam.detach().cpu().numpy(); cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    prob = torch.softmax(logits.detach(), 1)[0].cpu().numpy()
    return cim, cam, c, cmk, prob

## 1. Test predictions and confusion matrix

In [ ]:
linhas = []
for r in TESTE.itertuples():
    cim, cam, pred, cmk, prob = gradcam(os.path.join(DATASET_ORIGINAL, r.classe, r.arquivo))
    linhas.append(dict(classe=r.classe, arquivo=r.arquivo, real=CLASSES.index(r.classe), pred=pred, prob_pred=float(prob[pred]),
                       dentro_tolete=float(cam[cmk].sum() / (cam.sum() + 1e-8))))
P = pd.DataFrame(linhas); P['acertou'] = P.real == P.pred
acertos = int(P.acertou.sum())
print(f'Test accuracy: {acertos}/{len(P)} = {acertos/len(P):.4f}   (expected: 223/225 = 0.9911)')
if acertos != 223:
    print('WARNING: different from the expected value. A difference of one image can come from the epoch tie that defines best.pt or from rounding in the preprocessing; check that best.pt is the checkpoint of the reference run.')
P.to_csv(os.path.join(OUT, 'predicoes_teste.csv'), index=False)

cm = pd.crosstab(P.real, P.pred).reindex(index=range(len(CLASSES)), columns=range(len(CLASSES)), fill_value=0)
cm.index = [f'real {c}' for c in CLASSES]; cm.columns = [f'pred {c}' for c in CLASSES]
cm.to_csv(os.path.join(OUT, 'matriz_confusao.csv')); print(cm.to_string())

fig, ax = plt.subplots(figsize=(4.6, 4.0))
im = ax.imshow(cm.values, cmap='Blues')
ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels(CLASSES, rotation=35, ha='right', fontsize=8)
ax.set_yticks(range(len(CLASSES))); ax.set_yticklabels(CLASSES, fontsize=8)
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, int(cm.values[i, j]), ha='center', va='center', fontsize=9, color='white' if cm.values[i, j] > cm.values.max() / 2 else 'black')
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); plt.tight_layout()
plt.savefig(os.path.join(OUT, 'matriz_confusao.png'), dpi=300); plt.show()

print('\nMisclassified images:'); print(P[~P.acertou].assign(real=lambda d: [CLASSES[i] for i in d.real], pred=lambda d: [CLASSES[i] for i in d.pred])[['arquivo', 'real', 'pred', 'prob_pred']].to_string(index=False))

## 2. Learning curve

In [ ]:
log = pd.read_csv(os.path.join(RUN, 'treino', 'results.csv')); log.columns = [c.strip() for c in log.columns]
acc = log['metrics/accuracy_top1'].values; ep = log['epoch'].values if 'epoch' in log else np.arange(1, len(log) + 1)
if ep.min() == 0: ep = ep + 1
primeiro = int(acc.argmax()) + 1; ultimo = int(np.where(acc == acc.max())[0].max()) + 1
print(f'Epochs trained: {len(log)} | maximum validation accuracy: {acc.max():.4f} | first maximum: epoch {primeiro} | best.pt: epoch {ultimo} | tied epochs: {int((acc == acc.max()).sum())}')
fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
ax[0].plot(ep, log['train/loss'], label='Train loss'); ax[0].plot(ep, log['val/loss'], label='Validation loss')
ax[0].set_xlabel('Epoch'); ax[0].set_ylabel('Loss'); ax[0].legend(fontsize=8)
ax[1].plot(ep, acc, marker='o', ms=3, label='Validation top-1')
ax[1].axvline(ultimo, ls='--', c='gray', lw=0.8); ax[1].text(ultimo, acc.min(), f' saved model (epoch {ultimo})', fontsize=7, va='bottom')
ax[1].set_xlabel('Epoch'); ax[1].set_ylabel('Accuracy'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT, 'curva_aprendizado.png'), dpi=300); plt.show()
log.to_csv(os.path.join(OUT, 'results_seed_101.csv'), index=False)

## 3. Grad-CAM

In [ ]:
print(f"Share of the Grad-CAM activation inside the brightness mask (225 test images): mean={P.dentro_tolete.mean():.3f}  median={P.dentro_tolete.median():.3f}  min={P.dentro_tolete.min():.3f}")
print('(the mask excludes only near-black pixels and retains most of each crop, so this share is not a measure of the activation on the stalk; it is not used in the paper)')

rng = random.Random(0)
fig, axes = plt.subplots(len(CLASSES), N_PER_CLASS * 2, figsize=(3 * N_PER_CLASS * 2 * 0.9, 3 * len(CLASSES) * 0.9))
for r, cn in enumerate(CLASSES):
    itens = P[(P.real == r) & P.acertou].arquivo.tolist(); rng.shuffle(itens)
    for k, f in enumerate(itens[:N_PER_CLASS]):
        cim, cam, *_ = gradcam(os.path.join(DATASET_ORIGINAL, cn, f))
        a0, a1 = axes[r, 2 * k], axes[r, 2 * k + 1]
        a0.imshow(cim); a1.imshow(cim); a1.imshow(cam, cmap='jet', alpha=0.45)
        a0.set_title(cn if k == 0 else '', fontsize=9); a0.axis('off'); a1.axis('off')
plt.tight_layout(); plt.savefig(os.path.join(OUT, f'gradcam_seed_{SEED}.png'), dpi=300); plt.show()

E = P[~P.acertou].head(12)
if len(E):
    fig, axes = plt.subplots(len(E), 2, figsize=(4.2, 2.1 * len(E)), squeeze=False)
    for i, r in enumerate(E.itertuples()):
        cim, cam, *_ = gradcam(os.path.join(DATASET_ORIGINAL, r.classe, r.arquivo))
        axes[i, 0].imshow(cim); axes[i, 1].imshow(cim); axes[i, 1].imshow(cam, cmap='jet', alpha=0.45)
        axes[i, 0].set_title(f'true {r.classe}\npred {CLASSES[r.pred]}', fontsize=7); axes[i, 0].axis('off'); axes[i, 1].axis('off')
    plt.tight_layout(); plt.savefig(os.path.join(OUT, f'gradcam_erros_seed_{SEED}.png'), dpi=300); plt.show()
print('Files saved in', OUT)

## Part E: class-wise metrics and inference time in the 10 runs

For each seed: loads the `best.pt` of the run, predicts the 225 test images of that run (list in `split_agrupado_seed_<seed>.csv`) one at a time, after 10 warm-up images. The check of the number of correct predictions against `resumo_por_rodada.csv` must give `OK` in every run. Inference times come from the Ultralytics profiler; preprocessing includes reading and decoding the JPEG file and resizing it, because predictions are made from the file paths. Run it on the same GPU type and software environment that you want to report: absolute times depend on both.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
SEEDS = [42, 101, 2023, 777, 88, 1234, 555, 999, 333, 111]
RES = pd.read_csv(os.path.join(BASE_WORK_DIR, 'resumo_por_rodada.csv')).set_index('seed')
rows_pred, rows_cls, rows_lat = [], [], []
for seed in SEEDS:
    Sx = pd.read_csv(os.path.join(BASE_WORK_DIR, f'split_agrupado_seed_{seed}.csv'))
    Tx = Sx[Sx.particao == 'test'].sort_values(['classe', 'arquivo']).reset_index(drop=True)
    assert len(Tx) == 225
    mdl = YOLO(os.path.join(BASE_WORK_DIR, f'run_seed_{seed}', 'treino', 'weights', 'best.pt'))
    nomes = [mdl.names[i] for i in range(len(mdl.names))]; assert nomes == CLASSES, nomes
    caminhos = [os.path.join(DATASET_ORIGINAL, r.classe, r.arquivo) for r in Tx.itertuples()]
    for p in caminhos[:10]: mdl.predict(p, imgsz=224, verbose=False)                      # warm-up
    pre, inf, pos, pred = [], [], [], []
    for p in caminhos:
        r = mdl.predict(p, imgsz=224, verbose=False)[0]
        pred.append(int(r.probs.top1)); pre.append(r.speed['preprocess']); inf.append(r.speed['inference']); pos.append(r.speed['postprocess'])
    real = np.array([CLASSES.index(c) for c in Tx.classe]); pred = np.array(pred)
    ac = int((real == pred).sum()); esperado = int(RES.loc[seed, 'test_correct'])
    print(f'seed {seed:5d}: correct {ac}/225 | resumo_por_rodada.csv: {esperado} -> {"OK" if ac == esperado else "DIFFERENT"} | total latency {np.mean(pre)+np.mean(inf)+np.mean(pos):.2f} ms')
    P_, R_, F_, _ = precision_recall_fscore_support(real, pred, labels=range(len(CLASSES)), zero_division=0)
    for k, c in enumerate(CLASSES): rows_cls.append(dict(seed=seed, classe=c, precision=P_[k], recall=R_[k], f1=F_[k]))
    rows_cls.append(dict(seed=seed, classe='macro', precision=P_.mean(), recall=R_.mean(), f1=F_.mean()))
    rows_lat.append(dict(seed=seed, preprocess_ms=np.mean(pre), inference_ms=np.mean(inf), postprocess_ms=np.mean(pos), total_ms=np.mean(pre)+np.mean(inf)+np.mean(pos)))
    for r_, a, b in zip(Tx.itertuples(), real, pred): rows_pred.append(dict(seed=seed, classe=r_.classe, arquivo=r_.arquivo, real=int(a), pred=int(b)))
PR = pd.DataFrame(rows_pred); CL = pd.DataFrame(rows_cls); LT = pd.DataFrame(rows_lat)
PR.to_csv(os.path.join(OUT, 'predicoes_10_rodadas.csv'), index=False)
CL.to_csv(os.path.join(OUT, 'metricas_por_classe_por_rodada.csv'), index=False)
LT.to_csv(os.path.join(OUT, 'latencia_por_rodada.csv'), index=False)

print('\nCLASS-WISE METRICS (reference run, seed %d | mean ± SD over the 10 runs)' % SEED)
for c in CLASSES + ['macro']:
    ref = CL[(CL.seed == SEED) & (CL.classe == c)].iloc[0]; g = CL[CL.classe == c]
    print(f'{c:10s} ref P/R/F1 = {ref.precision:.3f} {ref.recall:.3f} {ref.f1:.3f} | mean±SD P = {g.precision.mean():.4f} ± {g.precision.std(ddof=1):.4f}  R = {g.recall.mean():.4f} ± {g.recall.std(ddof=1):.4f}  F1 = {g.f1.mean():.4f} ± {g.f1.std(ddof=1):.4f}')
print('\nMEAN LATENCY over the 10 runs (ms): pre %.2f | inference %.2f | post %.2f | total %.2f' % (LT.preprocess_ms.mean(), LT.inference_ms.mean(), LT.postprocess_ms.mean(), LT.total_ms.mean()))
try:
    import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (latency is NOT comparable with the paper)')
except Exception as e: print(e)
print('\nErrors per true cultivar, summed over the 10 runs:'); print(pd.crosstab(PR.real.map(lambda i: CLASSES[i]), PR.pred.map(lambda i: CLASSES[i])).to_string())

Output files, written to the `figuras_seed_101` folder: `predicoes_teste.csv`, `matriz_confusao.csv` and `.png`, `curva_aprendizado.png`, `results_seed_101.csv`, `gradcam_seed_101.png`, `gradcam_erros_seed_101.png`, `predicoes_10_rodadas.csv`, `metricas_por_classe_por_rodada.csv`, and `latencia_por_rodada.csv`.